In [1]:
# 1. Install the missing library in the cloud environment
!pip install faker

# 2. Run the data generation code
import pandas as pd
import numpy as np
import random
from faker import Faker

fake = Faker()
Faker.seed(42)
random.seed(42)

base_rows = 500
data = {
    'Customer ID': [1000 + i for i in range(base_rows)],
    'Full Name': [fake.name() for _ in range(base_rows)],
    'Age ': [random.randint(18, 70) for _ in range(base_rows)],
    'Gender': [random.choice(['M', 'F', 'Male', 'Female', 'm', 'f']) for _ in range(base_rows)],
    'Join Date': [],
    'Country Name': [random.choice(['USA', 'United States', 'US', 'UK', 'United Kingdom', 'Canada', ' Canada ']) for _ in range(base_rows)],
    'Annual Salary': [random.randint(30000, 120000) for _ in range(base_rows)]
}

date_formats = ['%Y-%m-%d', '%d-%m-%Y', '%Y/%m/%d', '%d/%m/%Y']
for _ in range(base_rows):
    dt = fake.date_between(start_date='-3y', end_date='today')
    fmt = random.choice(date_formats)
    data['Join Date'].append(dt.strftime(fmt))

df = pd.DataFrame(data)
df['Full Name'] = df['Full Name'].apply(lambda x: f"  {x}  " if random.random() < 0.15 else x)

for col in ['Age ', 'Join Date', 'Annual Salary']:
    df.loc[df.sample(frac=0.05).index, col] = np.nan

df.loc[df.sample(frac=0.03).index, 'Age '] = df['Age '].astype(str)

duplicates = df.sample(n=15, random_state=42)
df = pd.concat([df, duplicates], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save to CSV in cloud session storage
df.to_csv('messy_customer_data_500.csv', index=False)
print("File generated successfully inside Google Colab!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.1 MB/s eta 0:00:00
File generated successfully inside Google Colab!


/tmp/ipykernel_2021/2805515909.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['28.0' '30.0' '23.0' '30.0' '28.0' '38.0' '21.0' '61.0' '64.0' '32.0'
 '24.0' '31.0' '35.0' '63.0' '59.0']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df.sample(frac=0.03).index, 'Age '] = df['Age '].astype(str)


In [2]:
import pandas as pd
df = pd.read_csv('messy_customer_data_500.csv')
print("Original Shape:", df.shape)

Original Shape: (515, 7)


In [3]:
# Identify missing values in each column
missing_counts = df.isnull().sum()

print("Number of missing values per column:")
print(missing_counts)

Number of missing values per column:
Customer ID       0
Full Name         0
Age              27
Gender            0
Join Date        28
Country Name      0
Annual Salary    26
dtype: int64


In [4]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ' , '_')
print(df.columns)

Index(['customer_id', 'full_name', 'age', 'gender', 'join_date',
       'country_name', 'annual_salary'],
      dtype='object')


In [5]:
# To find count of duplicate rows

duplicate_rows = df[df.duplicated()]

print("Total Duplicate rows:", len(duplicate_rows), duplicate_rows.head)

Total Duplicate rows: 15 <bound method NDFrame.head of      customer_id           full_name   age  gender   join_date  \
33          1009          Ryan Munoz   NaN       f  2024/04/10   
38          1073      Lauren Daniels  20.0  Female  2026/03/05   
101         1124       Helen Jones     NaN  Female  26/03/2024   
102         1194       John Bishop    24.0       f  2025/08/13   
126         1155    Michelle Brown    27.0    Male  03-09-2025   
131         1084     Zachary Ferrell  58.0       M  14-12-2023   
205         1450        Brittany Kim  24.0       m  2025-12-17   
237         1394        Tamara Davis  20.0  Female  2025-03-17   
281         1361      Laurie Hoffman  27.0       f  2023/11/05   
299         1377       Paula Bradley  51.0  Female  26/09/2024   
329         1374     Charles Schultz  55.0    Male         NaN   
416         1104      Richard Henson  42.0       m         NaN   
458         1068         James Brown  41.0       m         NaN   
471         1406     

In [6]:
# To find duplicate pairs

all_duplicate_pairs = df[df.duplicated(keep=False)]
all_duplicate_pairs.sort_values(by='customer_id').head()

,customer_id,full_name,age,gender,join_date,country_name,annual_salary
7,1009,Ryan Munoz,NaN,f,2024/04/10,Canada,NaN
33,1009,Ryan Munoz,NaN,f,2024/04/10,Canada,NaN
41,1068,James Brown,41.0,m,NaN,US,48320.0
458,1068,James Brown,41.0,m,NaN,US,48320.0
38,1073,Lauren Daniels,20.0,Female,2026/03/05,United States,45220.0


In [7]:
# Remove duplicates
df.drop_duplicates(inplace=True)

In [8]:
Remaining_duplicates = df[df.duplicated()]
print("Total duplicates:", len(Remaining_duplicates))
print("New Dataset shape:", df.shape)

Total duplicates: 0
New Dataset shape: (500, 7)


In [9]:
# Force the age column to be numeric, turning text glitches into NaN blank spaces
df['age']= pd.to_numeric(df['age'],errors='coerce')

# Fill missing ages with the median age of the dataset
df['age'] = df['age'].fillna(df['age'].median())

# Fill missing annual salaries with the median salary
df['annual_salary'] = df['annual_salary'].fillna(df['annual_salary'].median())

In [16]:
# Convert salary to a clean integer type
df['annual_salary'] = df['annual_salary'].astype(int)

In [10]:
# convert age into int
df['age'] = df['age'].astype(int)

In [11]:
# Verify the changes by printing the data type of the column
print("The data type of age is now:", df['age'].dtype)
df[['age']].head()

The data type of age is now: int64


,age
0,61
1,53
2,69
3,21
4,20


In [12]:
# Remove any accidental spaces inside the text columns first
df['gender'] =df['gender'].str.strip()
df['country_name'] = df['country_name'].str.strip()

# Create maps that convert variations into standard names
gender_clean_map = {'M': 'Male', 'm': 'Male', 'Male': 'Male', 'F': 'Female', 'f': 'Female', 'Female': 'Female'}
country_clean_map = {'USA': 'USA', 'United States': 'USA', 'US': 'USA', 'UK': 'UK', 'United Kingdom': 'UK', 'Canada': 'Canada'}

# Apply the maps to overwrite the messy columns

df['gender'] =df['gender'].map(gender_clean_map)
df['country_name'] = df['country_name'].map(country_clean_map)

In [14]:
# Print unique values to confirm it worked
print("Unique Genders:", df['gender'].unique())
print("Unique Countries:", df['country_name'].unique())

Unique Genders: ['Female' 'Male']
Unique Countries: ['Canada' 'UK' 'USA']


In [15]:
# Convert all mixed date formats into a single, uniform standard timestamp format
df['join_date'] = pd.to_datetime(df['join_date'], errors= 'coerce')

# Drop any rows where the join_date couldn't be parsed or is completely missing
df.dropna(subset=['join_date'], inplace=True)

# Check the final row count
print("Final clean shape:", df.shape)

# Check the data type of the column
print("The data type of join_date is now:", df['join_date'].dtype)
df[['join_date']].head()

Final clean shape: (103, 7)
The data type of join_date is now: datetime64[ns]


,join_date
0,2025-02-04
3,2024-10-13
6,2023-07-28
8,2025-02-13
15,2026-06-17


In [17]:
# Save the final polished dataframe to a new CSV file
df.to_csv('cleaned_customer_data_500.csv', index=False)

In [18]:
print("Task 1 complete! Final dataset structure:")
df.info()

Task 1 complete! Final dataset structure:
<class 'pandas.core.frame.DataFrame'>
Index: 103 entries, 0 to 510
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customer_id    103 non-null    int64         
 1   full_name      103 non-null    object        
 2   age            103 non-null    int64         
 3   gender         103 non-null    object        
 4   join_date      103 non-null    datetime64[ns]
 5   country_name   103 non-null    object        
 6   annual_salary  103 non-null    int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 6.4+ KB
